# ✈️ 항공권 최저가 · 프로모션 비교 챗봇 (Gradio 6 + OpenAI)

**AI 업무자동화 과정 2-1 · 4일차 코랩 실습**

여행 조건을 문장으로 적으면 AI 가 조건을 읽고, 사이트별 검색 링크를 만들고, 웹 검색으로 프로모션 · 할인 쿠폰을 찾습니다.
그다음 **파이썬이 할인 조건을 따져 「할인 적용 후 최종 예상 금액」으로 순위**를 매깁니다.

### 시작하는 방법
1. 코랩 왼쪽 🔑 **보안 비밀**에 `OPENAI_API_KEY` 를 넣고 노트북 액세스를 켭니다.
2. **런타임 → 모두 실행**. ③ 셀 아래(또는 `https://….gradio.live`)에 화면이 뜹니다.
3. 화면의 탭 순서대로: **1. 여행 조건 → 2. 가격 비교 → 3. 프로모션 → 4. 최종 비교**

## 요청서를 어떻게 반영했나 · 그대로 못 한 것과 이유

| 요청 | 이 노트북에서 |
|---|---|
| 자연어 조건 읽기(출발 · 도착 · 날짜 · 인원 · 직항 · 수하물 · 예산 · 날짜 변경) · 모자라면 추가 질문 | AI 가 읽어 칸에 채우고, 빠진 것 · 지난 날짜 · 귀국일 오류는 **파이썬이 ❓ 질문**으로 안내 |
| 사이트별 가격 비교 · 실제 가격 없으면 검색 링크 | 스카이스캐너 · 네이버 항공권 · 카약 · 구글 항공편 · 트립닷컴은 **조건이 들어간 링크**, 나머지 여행사 · 항공사 10곳은 첫 화면 링크 |
| 프로모션 · 쿠폰 검색(공식 출처 · 확인 시각 · 종료된 것 제외) | OpenAI 웹 검색 → 표(할인 · 조건 · 중복 여부 · 기간 · 확인 상태 · 출처 URL · 확인 시각). **출처 URL 이 없는 것은 버림** |
| 할인 조건 검증 · 최종 결제 금액 · 중복 불가 쿠폰 · 「가능성」과 「확인」 구분 | 파이썬 계산: 기본가 + 세금 + 수수료 + 수하물 − 할인. **「실제 적용 확인」이고 「나에게 적용 = 예」인 할인만** 빼고, 최소 결제 · 최대 할인을 따지며, 중복 불가 쿠폰은 가장 큰 것 하나만 |
| 순위(최종 금액 → 기본가 → 직항) · 추천 · 기본가와 할인 후 순위 차이 · 주의사항 · 예약 링크 | 「4. 최종 비교」 |

**그대로 못 한 것과 이유**
| 요청 | 이렇게 했습니다 | 이유 |
|---|---|---|
| 각 사이트의 **실제 가격 자동 검색** | 링크를 열어 확인한 가격을 「2. 가격 비교」 표에 **직접 적습니다** | 네이버 · 스카이스캐너 · 여행사 · 항공사는 가격 자동 수집을 막고(로그인 · 봇 차단 · 이용약관), 가격 API 는 제휴 계약이 필요합니다. 가격을 지어내지 않으려고 사람이 확인한 값만 씁니다 |
| 「12월 중 가장 싼 날짜」 | 날짜 후보 2~3개를 정해 각각 링크로 비교하도록 안내 | 날짜별 가격표도 위와 같은 이유로 가져올 수 없습니다(스카이스캐너 · 구글 항공편의 「월 단위 보기」를 쓰면 편합니다) |
| 프로모션의 「실제 적용」 판정 | AI 가 공식 페이지에서 확인한 것만 「실제 적용 확인」, 나머지는 「적용 가능성 있음」. **나에게 해당하는지(신규 회원 · 앱 · 카드)는 사용자가 「예」로 고릅니다** | 회원 등급 · 카드 보유 · 앱 결제 여부는 프로그램이 알 수 없습니다 |

> 결과는 **예상 금액**입니다. 결제 직전 화면의 최종 금액과 쿠폰 조건을 꼭 확인하세요.

## ① 설치 (1분쯤)

In [ ]:
!pip install -q -U gradio openai pandas
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② API 키 불러오기

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ 보안 비밀에서 API 키를 읽었습니다.")
except Exception:
    print("보안 비밀에 키가 없습니다. 화면 위쪽 「API 키」 칸에 붙여 넣으세요.")

## ③ 챗봇 실행
멈추려면 ■(중지), 코드를 고쳤으면 이 셀만 다시 실행하세요.

In [ ]:
# ============================================================
# 항공권 최저가 · 프로모션 비교 챗봇 (Gradio 6 + OpenAI)
#  역할 나누기
#   - OpenAI: 여행 조건 읽기, 웹 검색으로 프로모션 찾기(출처 URL · 확인 시각)
#   - 파이썬: 사이트별 검색 링크 만들기, 할인 조건 검사, 최종 예상 금액 계산 · 순위
#  가격을 지어내지 않습니다. 사이트 가격은 사용자가 확인해 붙여 넣은 값만 씁니다.
# ============================================================
import os, re, json, inspect, datetime as dt
import urllib.parse as up
import pandas as pd
import gradio as gr
from openai import OpenAI

MODEL_DEFAULT = "gpt-4.1-mini"

# 자주 쓰는 도시 → 공항(도시) 코드. AI 가 코드를 못 주면 이 표로 채웁니다.
CITY = {"인천": "ICN", "서울": "SEL", "김포": "GMP", "제주": "CJU", "부산": "PUS", "김해": "PUS", "대구": "TAE", "청주": "CJJ",
        "광주": "KWJ", "도쿄": "TYO", "나리타": "NRT", "하네다": "HND", "오사카": "OSA", "간사이": "KIX", "후쿠오카": "FUK",
        "삿포로": "CTS", "오키나와": "OKA", "나고야": "NGO", "방콕": "BKK", "다낭": "DAD", "하노이": "HAN", "호치민": "SGN",
        "나트랑": "CXR", "세부": "CEB", "마닐라": "MNL", "타이베이": "TPE", "홍콩": "HKG", "싱가포르": "SIN", "괌": "GUM",
        "사이판": "SPN", "상하이": "SHA", "베이징": "BJS", "파리": "PAR", "런던": "LON", "뉴욕": "NYC", "로스앤젤레스": "LAX",
        "LA": "LAX", "시드니": "SYD", "발리": "DPS"}
DOMESTIC = {"ICN", "SEL", "GMP", "CJU", "PUS", "TAE", "CJJ", "KWJ"}

SITES_LINKED = ["스카이스캐너", "네이버 항공권", "카약", "구글 항공편", "트립닷컴"]
SITES_HOME = {"인터파크 투어": "https://travel.interpark.com/air", "하나투어": "https://www.hanatour.com/",
              "모두투어": "https://www.modetour.com/", "마이리얼트립": "https://www.myrealtrip.com/flights",
              "익스피디아": "https://www.expedia.co.kr/", "아고다": "https://www.agoda.com/ko-kr/flights",
              "대한항공": "https://www.koreanair.com/", "아시아나항공": "https://flyasiana.com/", "제주항공": "https://www.jejuair.net/",
              "진에어": "https://www.jinair.com/", "티웨이항공": "https://www.twayair.com/", "에어부산": "https://www.airbusan.com/",
              "에어서울": "https://flyairseoul.com/", "이스타항공": "https://www.eastarjet.com/", "에어프레미아": "https://www.airpremia.com/"}
ALL_SITES = SITES_LINKED + list(SITES_HOME)


# ---------- 1. OpenAI ----------
def get_client(key_box):
    key = (key_box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        raise gr.Error("OpenAI API 키가 없습니다. 위 「API 키」 칸에 붙여 넣거나 코랩 보안 비밀 OPENAI_API_KEY 에 넣어 주세요.")
    return OpenAI(api_key=key)

def friendly(e):
    s = str(e)
    if "invalid_api_key" in s or "Incorrect API key" in s:
        return "API 키가 맞지 않습니다."
    if "insufficient_quota" in s:
        return "OpenAI 계정 사용 한도(크레딧)가 없습니다. platform.openai.com 의 Billing 을 확인해 주세요."
    if "rate_limit" in s or "429" in s:
        return "요청이 너무 빨라 잠시 막혔습니다. 1분 뒤 다시 눌러 주세요."
    return "오류가 났습니다: " + s[:300]

def parse_json(raw):
    raw = (raw or "").strip()
    m = re.search(r"[\{\[].*[\}\]]", raw, re.S)
    try:
        return json.loads(m.group(0) if m else raw)
    except Exception:
        return None


# ---------- 2. 여행 조건 읽기 ----------
COND_PROMPT = """사용자의 여행 문장에서 항공권 검색 조건을 뽑아. 오늘은 {today} 이야. 연도가 없으면 오늘 이후 가장 가까운 날짜로 정해.
「3박 4일」처럼 기간만 있으면 귀국일 = 출발일 + 3일. 모르는 값은 null 로 두고 지어내지 마.
반드시 JSON 하나로만 답해:
{{"origin_city": "", "origin": "IATA 공항 또는 도시 코드", "dest_city": "", "dest": "IATA 코드",
 "depart": "YYYY-MM-DD", "return": "YYYY-MM-DD 또는 null", "trip": "왕복|편도", "adults": 1, "children": 0,
 "direct": "직항|경유 가능|상관없음", "bag": "필요|불필요|모름", "budget": null, "flexible": "가능|불가|모름",
 "month_only": "날짜 없이 달만 말했으면 YYYY-MM, 아니면 null"}}"""

def read_conditions(key_box, model, text):
    text = (text or "").strip()
    if not text:
        raise gr.Error("여행 조건을 적어 주세요. 예: 성인 2명, 인천에서 오사카 12월 5일부터 3박 4일 왕복")
    client = get_client(key_box)
    try:
        r = client.chat.completions.create(model=model, response_format={"type": "json_object"},
                                           messages=[{"role": "system", "content": COND_PROMPT.format(today=dt.date.today())},
                                                     {"role": "user", "content": text}])
    except Exception as e:
        raise gr.Error(friendly(e))
    c = parse_json(r.choices[0].message.content) or {}
    for k in ("origin", "dest"):
        if not c.get(k) and c.get(k + "_city"):
            c[k] = next((v for n, v in CITY.items() if n in c[k + "_city"]), None)
    return fill_form(c)

def fill_form(c):
    return (c.get("origin") or "", c.get("dest") or "", c.get("depart") or "", c.get("return") or "",
            c.get("trip") or "왕복", int(c.get("adults") or 1), int(c.get("children") or 0),
            c.get("direct") or "상관없음", c.get("bag") or "모름", c.get("budget") or None,
            check_missing(c.get("origin"), c.get("dest"), c.get("depart"), c.get("return"), c.get("trip") or "왕복",
                          c.get("adults"), c.get("month_only")))

def check_missing(o, d, dep, ret, trip, adults, month_only=None):
    """조건이 모자라면 무엇을 더 물어야 하는지 파이썬이 정합니다"""
    q = []
    if not o: q.append("어디에서 출발하시나요? (예: 인천, 김포, 부산)")
    if not d: q.append("어디로 가시나요?")
    if month_only and not dep: q.append(f"{month_only} 중 출발 날짜를 정해 주세요. 날짜를 바꿀 수 있다면 2~3개 후보로 각각 검색해 비교해 보세요.")
    elif not dep: q.append("출발 날짜는 언제인가요?")
    if trip == "왕복" and not ret: q.append("돌아오는 날짜는 언제인가요? (편도라면 「편도」로 바꿔 주세요)")
    if not adults: q.append("성인 몇 명인가요?")
    for name, v in (("출발일", dep), ("귀국일", ret)):
        if v:
            try:
                x = dt.date.fromisoformat(v)
                if x < dt.date.today(): q.append(f"{name}({v})이 오늘보다 앞입니다. 날짜를 확인해 주세요.")
            except ValueError:
                q.append(f"{name}을(를) 2026-12-05 처럼 적어 주세요.")
    if dep and ret and trip == "왕복":
        try:
            if dt.date.fromisoformat(ret) < dt.date.fromisoformat(dep): q.append("귀국일이 출발일보다 앞입니다.")
        except ValueError:
            pass
    return ("### ❓ 조금 더 알려 주세요\n" + "\n".join(f"- {x}" for x in q)) if q else "✅ 검색 조건이 모두 정해졌습니다. 「2. 가격 비교」 탭에서 링크를 열어 보세요."


# ---------- 3. 사이트별 검색 링크 (조건 반영) ----------
def links(o, d, dep, ret, trip, adults, children):
    o, d = (o or "").upper().strip(), (d or "").upper().strip()
    if not (o and d and dep):
        raise gr.Error("출발지 · 도착지 · 출발일을 먼저 정해 주세요.")
    rt = trip == "왕복" and ret
    a, c = int(adults or 1), int(children or 0)
    y = lambda s: s[2:4] + s[5:7] + s[8:10]
    nd = lambda s: s.replace("-", "")
    sky = f"https://www.skyscanner.co.kr/transport/flights/{o.lower()}/{d.lower()}/{y(dep)}/" + (f"{y(ret)}/" if rt else "") + f"?adultsv2={a}"
    kind = "domestic" if o in DOMESTIC and d in DOMESTIC else "international"
    nav = f"https://flight.naver.com/flights/{kind}/{o}-{d}-{nd(dep)}" + (f"/{d}-{o}-{nd(ret)}" if rt else "") + f"?adult={a}&child={c}&fareType=Y"
    kay = f"https://www.kayak.co.kr/flights/{o}-{d}/{dep}" + (f"/{ret}" if rt else "") + f"/{a}adults"
    gq = f"Flights from {o} to {d} on {dep}" + (f" through {ret}" if rt else " one way") + f" for {a} adults"
    goo = "https://www.google.com/travel/flights?hl=ko&q=" + up.quote(gq)
    trp = (f"https://kr.trip.com/flights/showfarefirst?dcity={o.lower()}&acity={d.lower()}&ddate={dep}"
           + (f"&rdate={ret}&triptype=rt" if rt else "&triptype=ow") + f"&class=y&quantity={a}")
    rows = [("스카이스캐너", sky), ("네이버 항공권", nav), ("카약", kay), ("구글 항공편", goo), ("트립닷컴", trp)]
    md = "#### 조건이 들어간 검색 링크 (새 창에서 열고 가격을 확인하세요)\n" + \
         "\n".join(f"- **{n}** — [{o}→{d} {dep}{(' ~ ' + ret) if rt else ''} · 성인 {a}]({u})" for n, u in rows) + \
         "\n\n#### 여행사 · 항공사 (첫 화면 — 같은 조건으로 직접 검색)\n" + \
         " · ".join(f"[{n}]({u})" for n, u in SITES_HOME.items()) + \
         "\n\n<small>사이트가 주소 형식을 바꾸면 조건이 빠진 첫 화면이 열릴 수 있습니다. 그때는 같은 조건으로 직접 검색해 주세요.</small>"
    return md

PRICE_COLS = ["예약 사이트", "기본 항공권 가격(원)", "유류할증료·세금(원)", "발권 수수료(원)", "위탁 수하물(원)",
              "직항(예/아니오)", "총 비행시간", "취소·변경 조건", "예약 링크", "확인 시각"]

def empty_prices():
    return pd.DataFrame([[s] + [""] * (len(PRICE_COLS) - 1) for s in ["스카이스캐너", "네이버 항공권", "트립닷컴", "제주항공", ""]],
                        columns=PRICE_COLS)


# ---------- 4. 프로모션 찾기 (웹 검색 · 출처 필수) ----------
PROMO_COLS = ["사이트", "프로모션", "할인 방식(정액/정률)", "할인 금액(원)", "할인율(%)", "최대 할인(원)", "최소 결제(원)",
              "쿠폰 코드", "조건", "중복 사용(예/아니오/모름)", "사용 기간", "탑승 기간", "확인 상태", "출처 URL", "확인 시각",
              "나에게 적용(예/아니오)"]

PROMO_PROMPT = """너는 항공권 할인 조사원이야. 웹 검색으로 아래 여행에 쓸 수 있는, 지금 진행 중인 항공권 프로모션 · 할인 쿠폰을 찾아.
[여행] {cond}
[찾을 곳] {sites}
- 여행사 · 항공사 공식 이벤트 페이지, 카드사 공식 이벤트 페이지를 우선 봐. 블로그 · 커뮤니티 글만 있으면 확인 상태를 「적용 가능성 있음」으로 해.
- 사용 기간이 오늘({today})을 지난 프로모션은 넣지 마.
- 숫자(할인 금액 · 할인율 · 최대 할인 · 최소 결제)는 출처에 적힌 그대로만 쓰고, 출처에 없으면 null.
- 확인 상태: 공식 페이지에서 이 노선 · 기간에 쓸 수 있음이 확인되면 「실제 적용 확인」, 아니면 「적용 가능성 있음」.
- 조건에는 신규 회원 · 앱 전용 · 특정 카드 · 첫 구매 · 노선 · 항공사 제한을 짧게 적어.
반드시 JSON 하나로만 답해(찾은 것이 없으면 빈 목록):
{{"promotions": [{{"site": "", "name": "", "type": "정액|정률", "amount": null, "rate": null, "max_discount": null,
 "min_amount": null, "code": "", "conditions": "", "stackable": "예|아니오|모름", "use_period": "", "travel_period": "",
 "status": "실제 적용 확인|적용 가능성 있음", "url": ""}}], "note": "찾지 못한 이유나 주의할 점"}}"""

def cond_text(o, d, dep, ret, trip, adults, children):
    return f"{o}→{d} · {trip} · 출발 {dep}" + (f" · 귀국 {ret}" if trip == "왕복" and ret else "") + f" · 성인 {adults} · 어린이 {children}"

def find_promos(key_box, model, o, d, dep, ret, trip, adults, children, sites):
    if not (o and d and dep):
        raise gr.Error("먼저 「1. 여행 조건」을 정해 주세요.")
    client = get_client(key_box)
    prompt = PROMO_PROMPT.format(cond=cond_text(o, d, dep, ret, trip, adults, children),
                                 sites=", ".join(sites or ALL_SITES), today=dt.date.today())
    now = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    try:
        r = client.responses.create(model=model, input=prompt, tools=[{"type": "web_search_preview"}])
        data = parse_json(r.output_text) or {}
    except Exception as e:
        s = str(e)
        if "tool" in s or "web_search" in s:
            return (pd.DataFrame(columns=PROMO_COLS),
                    "⚠️ **데이터 접근 제한** — 고른 모델로는 웹 검색을 쓸 수 없습니다. 모델을 gpt-4.1-mini 또는 gpt-4.1 로 바꾸거나, 아래 표에 직접 확인한 프로모션을 적어 주세요.")
        raise gr.Error(friendly(e))
    rows = []
    for p in data.get("promotions") or []:
        if not isinstance(p, dict) or not p.get("url"):
            continue                                   # 출처 없는 프로모션은 넣지 않습니다
        rows.append([p.get("site", ""), p.get("name", ""), p.get("type", ""), p.get("amount"), p.get("rate"),
                     p.get("max_discount"), p.get("min_amount"), p.get("code", ""), p.get("conditions", ""),
                     p.get("stackable", "모름"), p.get("use_period", ""), p.get("travel_period", ""),
                     p.get("status", "적용 가능성 있음"), p["url"], now, "아니오"])
    df = pd.DataFrame(rows, columns=PROMO_COLS)
    msg = (f"🔎 {len(rows)}건을 찾았습니다({now} 기준). **조건을 읽고 나에게 해당하면 「나에게 적용」을 「예」로 바꿔 주세요** "
           "(신규 회원 · 앱 결제 · 카드 보유 등). 「실제 적용 확인」이고 「예」인 것만 최종 금액에서 뺍니다.")
    if data.get("note"):
        msg += f"\n\n> {data['note']}"
    if not rows:
        msg = "검색에서 출처가 확인된 프로모션을 찾지 못했습니다. 각 사이트 이벤트 페이지를 직접 확인해 표에 적어 주세요." + \
              (f"\n\n> {data['note']}" if data.get("note") else "")
    return df, msg


# ---------- 5. 최종 금액 계산 · 순위 (파이썬) ----------
def num(v):
    s = re.sub(r"[^\d.]", "", str(v if v is not None else ""))
    try:
        return float(s) if s else None
    except ValueError:
        return None

def yes(v):
    return str(v or "").strip() in ("예", "Y", "y", "yes", "O", "o", "네")

def site_match(a, b):
    a, b = re.sub(r"\s", "", str(a or "")), re.sub(r"\s", "", str(b or ""))
    return bool(a and b and (a in b or b in a))

def discount_of(p, total):
    if p["min"] and total < p["min"]:
        return 0, f"최소 결제 {p['min']:,.0f}원 미만"
    d = p["amount"] if p["type"] == "정액" or (p["amount"] and not p["rate"]) else (total * p["rate"] / 100 if p["rate"] else 0)
    if not d:
        return 0, "할인 금액 정보 없음"
    if p["max"]:
        d = min(d, p["max"])
    return d, ""

def compare(prices, promos, need_bag, budget):
    pr = prices if isinstance(prices, pd.DataFrame) else pd.DataFrame(prices, columns=PRICE_COLS)
    po = promos if isinstance(promos, pd.DataFrame) else pd.DataFrame(promos, columns=PROMO_COLS)
    plist = []
    for _, r in po.iterrows():
        if not str(r.iloc[0]).strip():
            continue
        plist.append({"site": r.iloc[0], "name": r.iloc[1], "type": str(r.iloc[2]), "amount": num(r.iloc[3]), "rate": num(r.iloc[4]),
                      "max": num(r.iloc[5]), "min": num(r.iloc[6]), "stack": yes(r.iloc[9]), "status": str(r.iloc[12]),
                      "url": r.iloc[13], "mine": yes(r.iloc[15])})
    out, notes = [], []
    for _, r in pr.iterrows():
        site, base = str(r.iloc[0]).strip(), num(r.iloc[1])
        if not site or base is None:
            continue                                   # 가격을 넣지 않은 사이트는 계산하지 않습니다(지어내지 않음)
        tax, fee, bag = num(r.iloc[2]) or 0, num(r.iloc[3]) or 0, num(r.iloc[4]) or 0
        total = base + tax + fee + (bag if need_bag == "필요" else 0)
        cands = [p for p in plist if site_match(p["site"], site)]
        ok = [p for p in cands if p["mine"] and p["status"].startswith("실제")]
        maybe = [p for p in cands if not (p["mine"] and p["status"].startswith("실제"))]
        # 중복 불가 쿠폰은 가장 큰 것 하나만, 중복 가능한 것은 모두 더합니다
        best_single, best_d, stack_d, used = None, 0, 0, []
        for p in ok:
            dval, why = discount_of(p, total)
            if why:
                notes.append(f"- {site} · {p['name']}: {why}라 빼지 않았습니다.")
                continue
            if p["stack"]:
                stack_d += dval; used.append(f"{p['name']}(-{dval:,.0f})")
            elif dval > best_d:
                best_single, best_d = p, dval
        if best_single:
            used.append(f"{best_single['name']}(-{best_d:,.0f})")
        disc = min(best_d + stack_d, total)
        maybe_d = sum(discount_of(p, total)[0] for p in maybe)
        out.append({"예약 사이트": site, "기본 가격": base, "필수 부대비용": tax + fee + (bag if need_bag == "필요" else 0),
                    "할인 금액(확인)": disc, "최종 예상 금액": total - disc,
                    "적용 가능성 있음(미확인) 할인": maybe_d, "적용 할인": ", ".join(used) or "-",
                    "직항": "예" if yes(r.iloc[5]) else str(r.iloc[5] or "-"), "총 비행시간": r.iloc[6] or "-",
                    "취소·변경": r.iloc[7] or "-", "예약 링크": r.iloc[8] or "-", "확인 시각": r.iloc[9] or "-"})
    if not out:
        raise gr.Error("「2. 가격 비교」 표에 확인한 기본 항공권 가격을 한 줄 이상 넣어 주세요.")
    df = pd.DataFrame(out)
    df["_직항"] = df["직항"].map(lambda x: 0 if x == "예" else 1)
    df = df.sort_values(["최종 예상 금액", "기본 가격", "_직항"]).drop(columns="_직항").reset_index(drop=True)
    df.insert(0, "순위", range(1, len(df) + 1))
    top = df.iloc[0]
    cheapest_base = df.sort_values("기본 가격").iloc[0]
    rec = f"""### 🏆 추천: {top['예약 사이트']} — 최종 예상 **{top['최종 예상 금액']:,.0f}원**
- 기본 항공권 {top['기본 가격']:,.0f}원 + 필수 부대비용 {top['필수 부대비용']:,.0f}원 − 확인된 할인 {top['할인 금액(확인)']:,.0f}원
- 적용 할인: {top['적용 할인']}
- 예약 링크: {top['예약 링크']} · 가격 확인 시각: {top['확인 시각']}
"""
    if cheapest_base["예약 사이트"] != top["예약 사이트"]:
        rec += f"- 참고: 기본 가격만 보면 {cheapest_base['예약 사이트']}({cheapest_base['기본 가격']:,.0f}원)이(가) 가장 싸지만, 할인을 반영하면 순위가 바뀝니다.\n"
    if top["적용 가능성 있음(미확인) 할인"]:
        rec += f"- 미확인 할인 {top['적용 가능성 있음(미확인) 할인']:,.0f}원은 계산에서 뺐습니다. 공식 페이지에서 조건을 확인한 뒤 「확인 상태」를 「실제 적용 확인」으로 바꾸면 반영됩니다.\n"
    if budget and top["최종 예상 금액"] > float(budget):
        rec += f"- ⚠️ 희망 예산 {float(budget):,.0f}원보다 {top['최종 예상 금액'] - float(budget):,.0f}원 비쌉니다.\n"
    rec += ("\n**예약 시 주의** 표의 금액은 붙여 넣은 가격과 확인된 할인으로 계산한 **예상 금액**입니다. 결제 직전 화면의 최종 금액(세금 · 수수료 포함 여부), "
            "쿠폰 사용 조건, 카드 보유 여부, 취소 · 변경 수수료를 꼭 확인하세요.")
    if notes:
        rec += "\n\n**빼지 않은 할인**\n" + "\n".join(notes)
    return df, rec


# ---------- 6. 화면 ----------
HOWTO = """### 시작하는 방법
1. **여행 조건**을 문장으로 적고 「조건 읽기」 → 빈칸이나 ❓ 질문이 있으면 직접 채웁니다.
2. **가격 비교** 탭의 링크를 열어 사이트별 가격을 확인하고, 표에 **기본 가격 · 세금 · 수수료 · 수하물**을 적습니다.
3. **프로모션** 탭에서 「프로모션 찾기」(웹 검색) → 나에게 해당하는 할인의 「나에게 적용」을 「예」로.
4. **최종 비교** 탭에서 「계산」 → 할인 적용 후 최종 예상 금액 순위와 추천을 봅니다.

> 가격은 지어내지 않습니다. 사이트 가격은 내가 확인한 값만, 할인은 출처가 확인되고 나에게 해당하는 것만 계산에 넣습니다."""

THEME = gr.themes.Soft(primary_hue="sky", secondary_hue="indigo")
blocks_kwargs = {"title": "항공권 최저가 · 프로모션 비교"}
launch_kwargs = {"share": True, "debug": True}
if "theme" in inspect.signature(gr.Blocks.launch).parameters:
    launch_kwargs.update(theme=THEME)
else:
    blocks_kwargs.update(theme=THEME)

with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown("# ✈️ 항공권 최저가 · 프로모션 비교 챗봇")
    with gr.Accordion("시작하는 방법", open=True):
        gr.Markdown(HOWTO)
    with gr.Row():
        key_box = gr.Textbox(label="API 키 (보안 비밀에 넣었다면 비워 두기)", type="password", scale=3)
        model = gr.Dropdown(["gpt-4.1-mini", "gpt-4.1", "gpt-4o-mini"], value=MODEL_DEFAULT, label="모델",
                            allow_custom_value=True, scale=1)
    with gr.Tabs():
        with gr.Tab("1. 여행 조건"):
            text = gr.Textbox(label="여행 조건을 문장으로", lines=2,
                              placeholder="예: 성인 2명, 인천에서 오사카 12월 5일부터 3박 4일 왕복, 위탁 수하물 필요, 예산 60만 원")
            gr.Examples(["인천에서 도쿄 가는 11월 최저가 항공권 찾아줘.", "성인 2명, 오사카 왕복 3박 4일 항공권 찾아줘.",
                         "12월 중 일본 여행을 가려고 하는데 가장 저렴한 날짜를 추천해 줘.", "제주도 왕복 항공권 중 가장 저렴한 예약 사이트를 찾아줘."], text)
            read_btn = gr.Button("조건 읽기", variant="primary")
            with gr.Row():
                f_o = gr.Textbox(label="출발(공항 · 도시 코드)", placeholder="ICN")
                f_d = gr.Textbox(label="도착(코드)", placeholder="OSA")
                f_dep = gr.Textbox(label="출발일", placeholder="2026-12-05")
                f_ret = gr.Textbox(label="귀국일", placeholder="2026-12-08")
                f_trip = gr.Radio(["왕복", "편도"], value="왕복", label="여정")
            with gr.Row():
                f_a = gr.Number(label="성인", value=1, precision=0)
                f_c = gr.Number(label="어린이", value=0, precision=0)
                f_direct = gr.Radio(["직항", "경유 가능", "상관없음"], value="상관없음", label="직항")
                f_bag = gr.Radio(["필요", "불필요", "모름"], value="모름", label="위탁 수하물")
                f_budget = gr.Number(label="희망 예산(원, 선택)", precision=0)
            cond_msg = gr.Markdown()
            check_btn = gr.Button("조건 다시 확인", size="sm")
            gr.Markdown("<small>코드 예: 인천 ICN · 김포 GMP · 제주 CJU · 부산 PUS · 도쿄 TYO(나리타 NRT · 하네다 HND) · 오사카 OSA(간사이 KIX) · 후쿠오카 FUK · 방콕 BKK · 다낭 DAD</small>")

        with gr.Tab("2. 가격 비교"):
            link_btn = gr.Button("검색 링크 만들기", variant="primary")
            link_md = gr.Markdown()
            gr.Markdown("**확인한 가격을 표에 적어 주세요.** 사이트 가격에 세금이 이미 포함되어 있으면 「유류할증료·세금」은 0. 행이 모자라면 표 아래쪽에서 늘릴 수 있습니다.")
            price_tbl = gr.Dataframe(value=empty_prices(), headers=PRICE_COLS, interactive=True, wrap=True)

        with gr.Tab("3. 프로모션"):
            sites = gr.CheckboxGroup(ALL_SITES, value=ALL_SITES, label="찾을 사이트")
            promo_btn = gr.Button("프로모션 찾기 (웹 검색 · 30초~1분)", variant="primary")
            promo_msg = gr.Markdown()
            promo_tbl = gr.Dataframe(value=pd.DataFrame(columns=PROMO_COLS), headers=PROMO_COLS, interactive=True, wrap=True)

        with gr.Tab("4. 최종 비교"):
            calc_btn = gr.Button("최종 예상 금액 계산", variant="primary")
            rec_md = gr.Markdown()
            result_tbl = gr.Dataframe(label="할인 적용 후 순위", interactive=False, wrap=True)

    form = [f_o, f_d, f_dep, f_ret, f_trip, f_a, f_c]
    read_btn.click(read_conditions, [key_box, model, text], form + [f_direct, f_bag, f_budget, cond_msg])
    check_btn.click(lambda o, d, dep, ret, trip, a, c: check_missing(o, d, dep, ret, trip, a), form, cond_msg)
    link_btn.click(links, form, link_md)
    promo_btn.click(find_promos, [key_box, model] + form + [sites], [promo_tbl, promo_msg])
    calc_btn.click(compare, [price_tbl, promo_tbl, f_bag, f_budget], [result_tbl, rec_md])

if __name__ == "__main__":
    demo.launch(**launch_kwargs)

## 테스트 순서
1. 「성인 2명, 인천에서 오사카 12월 5일부터 3박 4일 왕복, 위탁 수하물 필요, 예산 60만 원」 → 조건 읽기 → 칸이 채워지는지
2. 「12월 중 일본 여행 가장 싼 날짜」 → ❓ 출발 날짜 · 도착지를 되묻는지
3. 2번 탭 「검색 링크 만들기」 → 링크를 열어 가격 확인 → 표에 2~3곳 적기
4. 3번 탭 「프로모션 찾기」 → 출처 URL 을 눌러 조건 확인 → 해당하는 것만 「나에게 적용」을 「예」로
5. 4번 탭 「계산」 → 할인 전후 순위가 다를 때 「참고」 문장이 나오는지, 미확인 할인이 따로 표시되는지

## 오류가 나면
| 증상 | 해결 |
|---|---|
| 「OpenAI API 키가 없습니다」 | 보안 비밀 이름 `OPENAI_API_KEY` · 노트북 액세스 → ② · ③ 다시 실행 |
| 「데이터 접근 제한 — 웹 검색을 쓸 수 없습니다」 | 모델을 `gpt-4.1-mini` 또는 `gpt-4.1` 로 |
| 링크를 열었는데 조건이 비어 있음 | 사이트가 주소 형식을 바꾼 것입니다. 같은 조건으로 직접 검색하세요 |
| 「가격을 한 줄 이상 넣어 주세요」 | 2번 탭 표의 「기본 항공권 가격」에 숫자를 적고 Enter |